In [91]:
from tensorflow.keras.models import load_model
model_load_path = r"C:\Users\asame\OneDrive\Desktop\tmp\model.keras"


# Load the model
lstm_model = load_model(model_load_path)

In [92]:
def process_multistream_features(frame_sequence):
    pose = frame_sequence[:, 21:54, :]       # Pose (33 landmarks)
    left_hand = frame_sequence[:, :21, :]    # Left hand (21 landmarks)
    right_hand = frame_sequence[:, 54:, :]   # Right hand (21 landmarks)
    
    return pose, left_hand, right_hand

In [93]:
# Target glosses (words)
target_glosses = ['any', 'thank you', 'bye', 'question']

# Create bidirectional mappings (label to gloss & gloss to label)
folder_to_label = {folder: idx for idx, folder in enumerate(target_glosses)}
label_to_folder = {idx: folder for folder, idx in folder_to_label.items()}

In [94]:
import mediapipe as mp
import cv2
import numpy as np
import os

mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

def extract_landmarks(image, hands, pose):
    # Initialize landmarks with zeros
    left_hand_landmarks = np.zeros((21, 3))
    right_hand_landmarks = np.zeros((21, 3))
    pose_landmarks = np.zeros((33, 3))
    
    with mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.5) as hands, \
         mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5) as pose:
        
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Process hands and pose
        hand_results = hands.process(image_rgb)
        pose_results = pose.process(image_rgb)

        # Extract hand landmarks
        if hand_results.multi_hand_landmarks:
            for idx, hand_landmarks_set in enumerate(hand_results.multi_hand_landmarks):
                extracted = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks_set.landmark])
                if hand_results.multi_handedness[idx].classification[0].label == 'Left':
                    left_hand_landmarks = extracted
                else:
                    right_hand_landmarks = extracted

        # Extract pose landmarks
        if pose_results.pose_landmarks:
            pose_landmarks = np.array([[lm.x, lm.y, lm.z] for lm in pose_results.pose_landmarks.landmark])
        
        # Combine all landmarks
        all_landmarks = np.vstack([left_hand_landmarks, pose_landmarks, right_hand_landmarks])
        #print(all_landmarks)
        hand_count = len(hand_results.multi_hand_landmarks) if hand_results.multi_hand_landmarks else 0
        pose_detected = 1 if pose_results.pose_landmarks else 0
        
        #print(all_landmarks.shape)
        return all_landmarks, hand_count, pose_detected

In [95]:
window_size = 30
confidence_threshold = 0.80
sliding_window = []
collecting = False

def predict_real_time(frame):
    global sliding_window, collecting

    landmarks, hand_count, _ = extract_landmarks(frame, mp_hands, mp_pose)

    if hand_count > 0:
        collecting = True
        sliding_window.append(landmarks)
        
        return ""

    elif collecting:
        collecting = False
        total_frames = len(sliding_window)
        if total_frames<20:
            return ""
        if total_frames < window_size:
            pad_count = window_size - total_frames
            pad_frame = np.zeros((75, 3))
            sampled_frames = sliding_window
            for i in range(pad_count):
                sampled_frames.append(pad_frame)
            print(len(sampled_frames))

        else:
            y = len(sliding_window)
            x = len(sliding_window) // 30
            sampled_frames = []
            pad_frame = np.zeros((75, 3))
            for i in range(0,y,x):
                if len(sampled_frames) < 30:
                    sampled_frames.append(sliding_window[i])
            while len(sampled_frames) < 30:
                sampled_frames.append(pad_frame)


        sequence = np.array(sampled_frames)
        sliding_window = []

        pose, left, right = process_multistream_features(sequence)

        predictions = lstm_model.predict([
            pose[np.newaxis, :],
            left[np.newaxis, :],
            right[np.newaxis, :]
        ])

        predicted_class = np.argmax(predictions, axis=1)[0]
        confidence = np.max(predictions)

        print(f"Confidence: {confidence:.2f}")
        if confidence > confidence_threshold:
            print(f"Prediction: {label_to_folder[predicted_class]}")
            return label_to_folder[predicted_class]

    return ""



In [96]:
cap = cv2.VideoCapture(0)  # Webcam feed
sentence = ""
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)  
    # Predict in real-time
    prediction = predict_real_time(frame)
    if prediction!= "":
        sentence= sentence + prediction + " "

    # Display predictions on the frame
    cv2.putText(frame, f"{sentence}", (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
    cv2.imshow("Sign Language Detection", frame)

    # Break loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 609ms/step
Confidence: 0.97
Prediction: any
30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Confidence: 0.52
30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Confidence: 0.99
Prediction: question
30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Confidence: 1.00
Prediction: bye
30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Confidence: 0.54
30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Confidence: 0.59
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Confidence: 1.00
Prediction: thank you
